# Seq2seq 한국어 번역기 만들기

In [1]:
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"사용 디바이스: {device}")
print("PyTorch 버전:", torch.__version__)

사용 디바이스: mps
PyTorch 버전: 2.14.0


In [2]:
# text->list(줄바꿈 기준)
ko_path = "korean-english-park.train.ko"
en_path = "korean-english-park.train.en"

with open(ko_path, "r", encoding="utf-8") as f:
    ko_lines = f.read().splitlines()

with open(en_path, "r", encoding="utf-8") as f:
    en_lines = f.read().splitlines()

In [3]:
print(len(ko_lines), len(en_lines))
print(ko_lines[0])
print(en_lines[0])

94123 94123
개인용 컴퓨터 사용의 상당 부분은 "이것보다 뛰어날 수 있느냐?"
Much of personal computing is about "can you top this?"


In [4]:
assert len(ko_lines) == len(en_lines)  # 두 파일의 줄 수가 같은지 먼저 확인

# (한국어, 영어) 튜플로 묶은 뒤 set으로 중복 제거
# 반드시 튜플 단위로 처리해야 병렬 쌍이 어긋나지 않음
paired = list(zip(ko_lines, en_lines))          # [(ko1,en1), (ko2,en2), ...] 형태로 묶기
cleaned_corpus = list(set(paired))              # set은 중복 허용 안 함 → 중복 쌍 제거

print(f"중복 제거 전: {len(paired)}쌍")
print(f"중복 제거 후: {len(cleaned_corpus)}쌍")

중복 제거 전: 94123쌍
중복 제거 후: 78968쌍


##### 

In [5]:
# 언어별 전처리 함수(이름 구분)
import re

def preprocess_sentence_en(sentence):
    """영어 문장 전처리: 노이즈 제거."""
    sentence = sentence.lower().strip()
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)      # 구두점 앞뒤에 공백 추가
    sentence = re.sub(r"[^a-z0-9?.!,]+", " ", sentence)     # 영어 알파벳, 숫자, 기본 구두점 외 전부 제거
    sentence = re.sub(r"\s+", " ", sentence)                # 연속 공백을 하나로 정리
    sentence = sentence.strip()                             # 맨 뒤에 붙어있는 공백(스페이스, 탭, 줄바꿈 등) 제거
    return sentence

def preprocess_sentence_ko(sentence):
    """한국어 문장 전처리: 완성형 한글(가-힣)을 허용 목록에 포함, 그 외 문자는 제거."""
    sentence = sentence.strip()
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)        # 구두점 앞뒤에 공백 추가
    sentence = re.sub(r'[" "]+', " ", sentence)               # 중복 공백 정리
    sentence = re.sub(r"[^가-힣0-9?.!, ]+", " ", sentence)    # 한글/숫자/기본 구두점 외 전부 제거
    sentence = re.sub(r"\s+", " ", sentence).strip()          # 정제 후 남은 중복 공백 재정리
    return sentence

In [6]:
# 전처리 적용 + 어절 수 40 이하 필터링(적절한 어절 수-평균값에서 확인)
kor_corpus = []
eng_corpus = []

for ko, en in cleaned_corpus:
    ko_clean = preprocess_sentence_ko(ko)   # 한국어 정제 함수 적용
    en_clean = preprocess_sentence_en(en)   # 영어 정제 함수 적용

    if not ko_clean or not en_clean:        # 정제 후 빈 문장이 된 경우 제외
        continue

    # 공백 기준 어절 수가 두 언어 모두 40 이하인 쌍만 선별
    if len(ko_clean.split()) <= 40 and len(en_clean.split()) <= 40:
        kor_corpus.append(ko_clean)
        eng_corpus.append(en_clean)

# 두 리스트 길이가 같은지, 인덱스가 서로 번역 쌍인지 확인
assert len(kor_corpus) == len(eng_corpus)

print(f"최종 데이터 개수: {len(kor_corpus)}쌍")
print("샘플 확인:")
for i in range(3):
    print(f"  KO: {kor_corpus[i]}")
    print(f"  EN: {eng_corpus[i]}")
    print()

최종 데이터 개수: 71486쌍
샘플 확인:
  KO: 주변 에 있는 물건들을 감지할 수 있는 5대의 카메라가 스크린 바로 위에 있다 .
  EN: five cameras that can sense nearby objects are mounted beneath the screen .

  KO: 브라질 환경보호당국은 16일 타파조스강에서 이 고래를 추적하다 놓쳐 고래에 대한 수색 작업을 중단했다 .
  EN: brazil s environmental protection agency had called off its search for the whale late friday after losing track of the mammal in the tapajos river .

  KO: 부시 미 대통령은 이라크 전쟁에 부적합한 장비가 거론되는 것에 대해 우려를 표명하고 도날드 럼스펠드 국방장관에게 문제를 제기한 병사들을 비난하지는 않는다고 목요일날 밝혔다 .
  EN: president bush said on thursday u . s . troop concerns about inadequate equipment for iraq combat are being addressed and he did not blame soldiers for raising the issue with defense secretary donald rumsfeld .



##### 

In [7]:
# vocab_size / max_len 실험
import sentencepiece as spm
import numpy as np

# 임시로 작은 vocab으로 한 번 학습해서 길이 분포만 확인 (실험용)
with open("kor_tmp.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(kor_corpus))
with open("eng_tmp.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(eng_corpus))

# 학습된 토크나이저 불러오기, 문장 수 세기
spm.SentencePieceTrainer.train(input="kor_tmp.txt", model_prefix="kor_tmp", vocab_size=16000)
spm.SentencePieceTrainer.train(input="eng_tmp.txt", model_prefix="eng_tmp", vocab_size=16000)

kor_tmp_sp = spm.SentencePieceProcessor(); kor_tmp_sp.load("kor_tmp.model")
eng_tmp_sp = spm.SentencePieceProcessor(); eng_tmp_sp.load("eng_tmp.model")

kor_lens = [len(kor_tmp_sp.encode(s)) for s in kor_corpus] # 문장마다 토큰 개수를 리스트로 만듦
eng_lens = [len(eng_tmp_sp.encode(s)) for s in eng_corpus]

print("한국어 - 평균:", np.mean(kor_lens), "95%:", np.percentile(kor_lens, 95), "최대:", max(kor_lens))
print("영어   - 평균:", np.mean(eng_lens), "95%:", np.percentile(eng_lens, 95), "최대:", max(eng_lens))

I0000 00:00:1788844152.046963 1050837 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: kor_tmp.txt
  input_format: 
  model_prefix: kor_tmp
  model_type: UNIGRAM
  vocab_size: 16000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_dif

한국어 - 평균: 22.565159611672215 95%: 39.0 최대: 84
영어   - 평균: 24.305234591388523 95%: 40.0 최대: 59


| 구분 | 평균 | 95% 지점 | 최대 |
|---|---|---|---|
| 한국어 | 22.6 | 39 | 84 |
| 영어 | 24.3 | 40 | 59 |

In [8]:
# 토크나이저
import torch

def tokenize(corpus, prefix, vocab_size=16000, max_len=40,
             pad_id=0, bos_id=1, eos_id=2, unk_id=3):
    
    #corpus(문장 리스트)를 받아 SentencePiece를 학습하고,
    #(텐서, tokenizer)를 반환하는 함수
    
    # 1. corpus를 파일로 저장 (SentencePiece는 파일 입력만 받음)
    txt_path = f"{prefix}_corpus.txt"
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write("\n".join(corpus))

    # 2. SentencePiece 학습
    spm.SentencePieceTrainer.train(
        input=txt_path,
        model_prefix=prefix,
        vocab_size=vocab_size,
        pad_id=pad_id, bos_id=bos_id, eos_id=eos_id, unk_id=unk_id
    )

    # 3. 학습된 tokenizer 불러오기
    tokenizer = spm.SentencePieceProcessor()
    tokenizer.load(f"{prefix}.model")

    # 4. 문장 -> bos + 토큰ID + eos -> 패딩까지 적용해 텐서로 변환
    tensor = []
    for s in corpus:
        ids = [bos_id] + tokenizer.encode(s) + [eos_id]
        ids = ids[:max_len]                              # 최대 길이 초과 시 자르기
        ids = ids + [pad_id] * (max_len - len(ids))       # 부족하면 패딩
        tensor.append(ids)

    tensor = torch.tensor(tensor)
    return tensor, tokenizer

In [9]:
vocab_size = 16000   # 최소 10000 이상 
max_len = 40          # 위 길이 분포 확인 후 조정

kor_tensor, encoder_tokenizer = tokenize(kor_corpus, "encoder_spm", vocab_size=vocab_size, max_len=max_len)
eng_tensor, decoder_tokenizer = tokenize(eng_corpus, "decoder_spm", vocab_size=vocab_size, max_len=max_len)

print("한국어(인코더) vocab 크기:", len(encoder_tokenizer))
print("영어(디코더) vocab 크기:", len(decoder_tokenizer))
print("kor_tensor shape:", kor_tensor.shape)   # (71486, max_len)
print("eng_tensor shape:", eng_tensor.shape)   # (71486, max_len)
print(kor_tensor[0])
print(eng_tensor[0])

I0000 00:00:1788844182.610158 1050837 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: encoder_spm_corpus.txt
  input_format: 
  model_prefix: encoder_spm
  model_type: UNIGRAM
  vocab_size: 16000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 3
  bos_id: 1
  eos_id: 2
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  �

한국어(인코더) vocab 크기: 16000
영어(디코더) vocab 크기: 16000
kor_tensor shape: torch.Size([71486, 40])
eng_tensor shape: torch.Size([71486, 40])
tensor([   1,  915,  100,   32, 5466,  120, 4303,   38,   24,   32,   68, 1590,
        2807,   12, 6564, 1007, 5016,   19,    4,    2,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0])
tensor([   1,  187, 3893,   17,   95, 2335, 1289, 4780,   32, 8859, 4459,    4,
        1923,    5,    2,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0])


In [10]:
# eos_id(2)가 각 문장에 실제로 포함되어 있는지 비율 확인
eos_included_kor = (kor_tensor == 2).any(dim=1).float().mean()
eos_included_eng = (eng_tensor == 2).any(dim=1).float().mean()

print(f"한국어 텐서 중 eos 포함 비율: {eos_included_kor:.2%}")
print(f"영어 텐서 중 eos 포함 비율: {eos_included_eng:.2%}")

한국어 텐서 중 eos 포함 비율: 94.20%
영어 텐서 중 eos 포함 비율: 92.82%


In [11]:
max_len = 45   # 40 → 45로 변경, bos/eos 여유분 확보

kor_tensor, encoder_tokenizer = tokenize(kor_corpus, "encoder_spm", vocab_size=16000, max_len=max_len)
eng_tensor, decoder_tokenizer = tokenize(eng_corpus, "decoder_spm", vocab_size=16000, max_len=max_len)

print("kor_tensor shape:", kor_tensor.shape)
print("eng_tensor shape:", eng_tensor.shape)

eos_included_kor = (kor_tensor == 2).any(dim=1).float().mean()
eos_included_eng = (eng_tensor == 2).any(dim=1).float().mean()
print(f"한국어 eos 포함 비율: {eos_included_kor:.2%}")
print(f"영어 eos 포함 비율: {eos_included_eng:.2%}")

I0000 00:00:1788844249.778460 1050837 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: encoder_spm_corpus.txt
  input_format: 
  model_prefix: encoder_spm
  model_type: UNIGRAM
  vocab_size: 16000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 3
  bos_id: 1
  eos_id: 2
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  �

kor_tensor shape: torch.Size([71486, 45])
eng_tensor shape: torch.Size([71486, 45])
한국어 eos 포함 비율: 97.62%
영어 eos 포함 비율: 98.74%


| 구분 | max_len=40 | max_len=45 |
|---|---|---|
| 한국어 eos 포함 비율 | 94.20% | 97.62% |
| 영어 eos 포함 비율 | 92.82% | 98.74% |

###### -> max_len=40으로 잘려서 eos가 소실됐던 문장 (~5~8%) 오히려 잘려나갔던 내용(eos 포함)이 복원됨. 

In [12]:
print("최종 확정된 설정")
print("vocab_size:", 16000)
print("max_len:", 45)
print("kor_tensor shape:", kor_tensor.shape)
print("eng_tensor shape:", eng_tensor.shape)
print("encoder_tokenizer vocab:", len(encoder_tokenizer))
print("decoder_tokenizer vocab:", len(decoder_tokenizer))

최종 확정된 설정
vocab_size: 16000
max_len: 45
kor_tensor shape: torch.Size([71486, 45])
eng_tensor shape: torch.Size([71486, 45])
encoder_tokenizer vocab: 16000
decoder_tokenizer vocab: 16000


##### 

In [13]:
import torch
import torch.nn as nn

# 공통 설정
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
input_dim = len(encoder_tokenizer)
output_dim = len(decoder_tokenizer)
emb_dim = 256
hid_dim = 512
pad_id, bos_id, eos_id = 0, 1, 2


class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hidden_dim, batch_first=True)

    def forward(self, src):                       # src: (batch, src_len)
        embedded = self.embedding(src)
        outputs, hidden = self.rnn(embedded)        # outputs: (batch, src_len, hidden_dim)
        return outputs, hidden


class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.W1 = nn.Linear(hidden_dim, hidden_dim)
        self.W2 = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        hidden = hidden[-1].unsqueeze(1)            # (batch, 1, hid)
        energy = torch.tanh(self.W1(encoder_outputs) + self.W2(hidden))
        score = self.v(energy).squeeze(2)           # (batch, src_len)
        return torch.softmax(score, dim=1)


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, attention):
        super().__init__()
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim + hidden_dim, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, input, hidden, encoder_outputs):
        embedded = self.embedding(input).unsqueeze(1)          # (batch, 1, emb_dim)
        attn_weights = self.attention(hidden, encoder_outputs)  # (batch, src_len)
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs)  # (batch, 1, hid)

        rnn_input = torch.cat((embedded, context), dim=2)
        output, hidden = self.rnn(rnn_input, hidden)

        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, attn_weights


class Seq2SeqAttention(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg=None, max_len=45, bos_id=1, eos_id=2):
        batch_size = src.shape[0]
        encoder_outputs, hidden = self.encoder(src)
        outputs, attentions = [], []

        if trg is not None:
            for t in range(trg.shape[1]):
                input = trg[:, t]
                output, hidden, attn = self.decoder(input, hidden, encoder_outputs)
                outputs.append(output.unsqueeze(1))
                attentions.append(attn.unsqueeze(1))
        else:
            input = torch.full((batch_size,), bos_id, dtype=torch.long, device=self.device)
            finished = torch.zeros(batch_size, dtype=torch.bool, device=self.device)
            for t in range(max_len):
                output, hidden, attn = self.decoder(input, hidden, encoder_outputs)
                outputs.append(output.unsqueeze(1))
                attentions.append(attn.unsqueeze(1))
                input = output.argmax(1)
                finished |= (input == eos_id)
                if finished.all():
                    break

        return torch.cat(outputs, dim=1), torch.cat(attentions, dim=1)


def build_model(attention_class):
    encoder = Encoder(input_dim, emb_dim, hid_dim).to(device)
    attention = attention_class(hid_dim).to(device)
    decoder = Decoder(output_dim, emb_dim, hid_dim, attention).to(device)
    return Seq2SeqAttention(encoder, decoder, device).to(device)


model_bahdanau = build_model(BahdanauAttention)
print(model_bahdanau)

Seq2SeqAttention(
  (encoder): Encoder(
    (embedding): Embedding(16000, 256)
    (rnn): GRU(256, 512, batch_first=True)
  )
  (decoder): Decoder(
    (attention): BahdanauAttention(
      (W1): Linear(in_features=512, out_features=512, bias=True)
      (W2): Linear(in_features=512, out_features=512, bias=True)
      (v): Linear(in_features=512, out_features=1, bias=False)
    )
    (embedding): Embedding(16000, 256)
    (rnn): GRU(768, 512, batch_first=True)
    (fc_out): Linear(in_features=512, out_features=16000, bias=True)
  )
)


In [ ]:
# 모델 학습시키기

In [22]:
from torch.utils.data import TensorDataset, DataLoader

# eng_tensor는 [bos, 토큰들..., eos, pad...] 형태이므로
# 한 칸씩 밀어서 디코더 입력/타겟(teacher forcing용)으로 나눔
trg_input = eng_tensor[:, :-1]
trg_label = eng_tensor[:, 1:]

dataset = TensorDataset(kor_tensor, trg_input, trg_label)

BATCH_SIZE = 128   # 64 → 128로 변경
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"전체 배치 수: {len(train_loader)}")  

전체 배치 수: 559


### Batch size 변화 64->128 비교

| 구분 | BATCH_SIZE=64 | BATCH_SIZE=128 |
|---|---|---|
| 전체 데이터 | 71,486개 | 71,486개 (동일) |
| 배치 수 계산 | 71,486 ÷ 64 | 71,486 ÷ 128 |
| 전체 배치 수 | 1,117 | 559 (거의 절반) |

#### 

In [23]:
#옵티마이저
model_bahdanau = build_model(BahdanauAttention)
optimizer = optim.Adam(model_bahdanau.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=pad_id)

print(model_bahdanau)

Seq2SeqAttention(
  (encoder): Encoder(
    (embedding): Embedding(16000, 256)
    (rnn): GRU(256, 512, batch_first=True)
  )
  (decoder): Decoder(
    (attention): BahdanauAttention(
      (W1): Linear(in_features=512, out_features=512, bias=True)
      (W2): Linear(in_features=512, out_features=512, bias=True)
      (v): Linear(in_features=512, out_features=1, bias=False)
    )
    (embedding): Embedding(16000, 256)
    (rnn): GRU(768, 512, batch_first=True)
    (fc_out): Linear(in_features=512, out_features=16000, bias=True)
  )
)


##### 

In [24]:
from tqdm import tqdm

def train_step(model, data_loader, optimizer, criterion, epoch):
    model.train()
    epoch_loss = 0

    progress_bar = tqdm(data_loader, desc=f"Epoch {epoch+1}", leave=True)

    for src, ti, tl in progress_bar:
        src, ti, tl = src.to(device), ti.to(device), tl.to(device)   # permute 없이 그대로 device로만 이동

        optimizer.zero_grad()
        outputs, _ = model(src, ti)                        # outputs: (batch, trg_len, vocab)

        outputs = outputs.reshape(-1, outputs.shape[-1])
        tl = tl.reshape(-1)

        loss = criterion(outputs, tl)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)
        optimizer.step()

        epoch_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

    return epoch_loss / len(data_loader)

In [17]:
def evaluate(sentence, model, encoder_tokenizer, decoder_tokenizer, max_len=45):
    model.eval()
    sentence = preprocess_sentence_ko(sentence)

    src_ids = encoder_tokenizer.encode(sentence)[:max_len]
    src_pieces = [encoder_tokenizer.id_to_piece(i) for i in src_ids]
    src_ids = src_ids + [pad_id] * (max_len - len(src_ids))
    src_tensor = torch.tensor(src_ids).unsqueeze(0).to(device)   # ← (1, src_len)으로 수정!

    with torch.no_grad():
        outputs, attentions = model(src_tensor, max_len=max_len)  # outputs: (1, trg_len, vocab)

    pred_ids = outputs.argmax(2).squeeze(0).tolist()               # ← squeeze(0)으로 수정!
    if eos_id in pred_ids:
        pred_ids = pred_ids[:pred_ids.index(eos_id)]

    result_pieces = [decoder_tokenizer.id_to_piece(i) for i in pred_ids]
    result_text = decoder_tokenizer.decode(pred_ids)
    attention = attentions.squeeze(0).cpu().numpy()[:len(pred_ids), :len(src_pieces)]  # ← squeeze(0)

    return result_text, result_pieces, src_pieces, attention


def translate(sentence, model, encoder_tokenizer, decoder_tokenizer, max_len=45):
    result_text, result_pieces, src_pieces, attention = evaluate(
        sentence, model, encoder_tokenizer, decoder_tokenizer, max_len
    )
    print(f"  {sentence} → {result_text}")
    return result_text, result_pieces, src_pieces, attention

In [25]:
# 1. device 자체가 mps로 잡혔는지
print("device 변수:", device)

# 2. 모델 파라미터가 실제로 mps에 올라가 있는지
print("모델 파라미터 위치:", next(model_bahdanau.parameters()).device)

# 3. 실제 연산 중 텐서가 mps에서 도는지 (배치 하나 테스트)
src, ti, tl = next(iter(train_loader))
src, ti, tl = src.to(device), ti.to(device), tl.to(device)
print("입력 텐서 위치:", src.device)

outputs, _ = model_bahdanau(src, ti)
print("출력 텐서 위치:", outputs.device)

device 변수: mps
모델 파라미터 위치: mps:0
입력 텐서 위치: mps:0
출력 텐서 위치: mps:0


In [27]:

test_sentences = [
    "오바마는 대통령이다.",
    "시민들은 도시 속에 산다.",
    "커피는 필요 없다.",
    "일곱 명의 사망자가 발생했다."
]

EPOCHS = 15
loss_history = []

for epoch in range(EPOCHS):
    train_loss = train_step(model_bahdanau, train_loader, optimizer, criterion, epoch)
    loss_history.append(train_loss)
    print(f"Epoch {epoch+1}/{EPOCHS}, Train Loss: {train_loss:.4f}")

    for s in test_sentences:
        translate(s, model_bahdanau, encoder_tokenizer, decoder_tokenizer, max_len=45)
    print()

Epoch 1: 100%|████████████████████| 559/559 [07:45<00:00,  1.20it/s, loss=4.75]


Epoch 1/15, Train Loss: 4.9549
  오바마는 대통령이다. → obama s campaign is expected to be the president .
  시민들은 도시 속에 산다. → the last few days of the year , the world s largest archipelago is a .
  커피는 필요 없다. → it s not a very good thing .
  일곱 명의 사망자가 발생했다. → a suicide bomber killed at least three people dead .



Epoch 2: 100%|████████████████████| 559/559 [08:10<00:00,  1.14it/s, loss=4.28]


Epoch 2/15, Train Loss: 4.3751
  오바마는 대통령이다. → obama , obama , obama , who is in the world , he said .
  시민들은 도시 속에 산다. → the city s most famous cities of the city s most populous city .
  커피는 필요 없다. → it s not just a few weeks .
  일곱 명의 사망자가 발생했다. → a suicide bomber killed at least 30 people .



Epoch 3: 100%|████████████████████| 559/559 [07:24<00:00,  1.26it/s, loss=3.85]


...

Epoch 13/15, Train Loss: 1.6763
  오바마는 대통령이다. → obama will be president , obama said , but he ll still be president .
  시민들은 도시 속에 산다. → this is a tiny known event in the city s cramerme .
  커피는 필요 없다. → it s not all the way it s been built like anything just one of the few .
  일곱 명의 사망자가 발생했다. → seven people were killed when a suicide bomber was killed .



Epoch 14: 100%|███████████████████| 559/559 [08:19<00:00,  1.12it/s, loss=1.82]


Epoch 14/15, Train Loss: 1.5742
  오바마는 대통령이다. → obama will be running in , obama said he would be president , he will be .
  시민들은 도시 속에 산다. → this is a highly draw at urban where city is a bigger hub .
  커피는 필요 없다. → it s not easy to pull out the old cathon .
  일곱 명의 사망자가 발생했다. → eight people were killed when a suicide bomber at 66 year old .



Epoch 15: 100%|███████████████████| 559/559 [07:45<00:00,  1.20it/s, loss=1.57]

Epoch 15/15, Train Loss: 1.4849
  오바마는 대통령이다. → obama will be president , he said obama will be president , he will .
  시민들은 도시 속에 산다. → this is a highly rising shaped line between the city s pwack and urbanp .
  커피는 필요 없다. → it s not easy to be leaned .
  일곱 명의 사망자가 발생했다. → seven people were killed when a suicide bomber at 69 , she said .



### ⛳️ 15회 학습 후 결과

| 항목 | 값 |
|---|---|
| 총 소요 시간 | 1시간 52분 15초 (6,735초) |
| 평균 에폭 시간 | 약 7분 29초 (449초) |
| 시작 Loss (Epoch 1) | 4.9549 |
| 최종 Loss (Epoch 15) | 1.4849 |
| Loss 감소량 | 3.47 |
| Loss 감소율 | 약 70.0% |

### k1-k4 최고, 최저 번역 

| 예문 (한국어) | 구분 | 회차 | 영어 번역 결과 |
|---|---|---|---|
| K1) 오바마는 대통령이다. | 최고 | Epoch 8 | obama will be the president , obama said . |
| | 최저 | Epoch 3 | obama , obama , obama , the illinois senator , obama , and his president is to be in . |
| K2) 시민들은 도시 속에 산다. | 최고 | Epoch 4 | the city s urban region is the second smallest of the city . |
| | 최저 | Epoch 15 | this is a highly rising shaped line between the city s pwack and urbanp . |
| K3) 커피는 필요 없다. | 최고 | Epoch 1 | it s not a very good thing . |
| | 최저 | Epoch 13 | it s not all the way it s been built like anything just one of the few . |
| K4) 일곱 명의 사망자가 발생했다. | 최고 | Epoch 13 | seven people were killed when a suicide bomber was killed . |
| | 최저 | Epoch 11 | eight people were killed when a suicide bomber apparently at least 60 people who were home to the death of a magnitude . |

### k1-k4 번역의 특징 

| 특징 | 최고 번역 | 최저 번역 |
|---|---|---|
| 문장 길이 | 짧음 (7~11단어) | 김 (15단어 이상) |
| 절 연결 방식 | 한 번에 깔끔히 종결 | 쉼표/관계절로 계속 이어붙임 |
| 단어 반복 | 없음 | 특정 단어(예: obama)가 반복되며 붕괴 시작 |
| 문장 유형 | 정형화된 뉴스 패턴(사건+숫자) 문장에서 강세 | 추상적/일반적 진술 문장에서 약세 |

####

## ⛳️ 랜덤 문장 다섯 개 평가하기

#### 평가기준: 정성적 평가(Qualitative Assessment), 정량적 평가(Quantative Assesment)

### 정성적 평가 기준

| 평가 항목 | 확인할 질문 | 좋음(●) | 보통(◐) | 나쁨(○) |
|---|---|---|---|---|
| 의미 전달 | 원문의 핵심 의미(누가, 무엇을, 어떻게)가 전달되는가? | 핵심 의미가 정확히 전달됨 | 일부만 전달됨 | 의미 전달 실패 |
| 문법 정확성 | 문장이 영어 문법상 자연스러운가? | 문법 오류 없음 | 어색하지만 이해 가능 | 문법이 심하게 깨짐 |
| 문장 구조/어순 | 원문의 구조(주어-동사-목적어 등)가 잘 반영됐는가? | 구조가 잘 대응됨 | 구조가 다소 어긋남 | 구조 자체가 무너짐 |
| 반복/붕괴 현상 | 같은 단어·구가 불필요하게 반복되지 않는가? | 반복 없음 | 경미한 반복 | 심한 반복(루프) |
| 숫자/고유명사 정확도 | 숫자, 이름 등이 정확히 옮겨졌는가? | 정확히 일치 | 근사치/부분 일치 | 완전히 다른 값 |
| 문장 종결 | 문장이 자연스럽게 끝맺는가? (eos 잘 생성) | 자연스럽게 종결 | 다소 급작스럽게 끝남 | 끝없이 이어지거나 중간에 끊김 |
| 유창성(Fluency) | 원문을 몰라도 이 영어 문장 자체가 자연스럽게 읽히는가? | 원어민 수준으로 자연스러움 | 어색하지만 읽을 수 있음 | 문장 자체가 말이 안 됨 |

### 정량적 평가 기준-BLEU 점수 기준(일반적으로 통용되는 가이드)

| BLEU 점수 범위 | 품질 수준 | 설명 |
|---|---|---|
| < 0.10 | 거의 무의미한 번역 | 이해하기 어려운 수준, 원문과 관련성 낮음 |
| 0.10 ~ 0.19 | 핵심 주제만 파악 가능 | 일부 단어/구는 일치하나 전체 의미 전달은 어려움 |
| 0.20 ~ 0.29 | 대의는 전달되나 문법적으로 어색함 | 주요 내용은 통하지만 다듬어지지 않은 번역 |
| 0.30 ~ 0.39 | 이해 가능한 수준의 번역 | 실용적으로 쓸 수 있는 최소 수준 |
| 0.40 ~ 0.49 | 높은 품질의 번역 | 유창하고 정확한 번역 |
| 0.50 이상 | 사람 번역 수준에 근접 | 전문 번역가 수준, 상용 번역기(구글 번역 등)가 이 구간 |
| 1.00 | 정답과 완전히 동일 | reference와 100% 일치 (n-gram까지 전부 겹침) |

##### 

In [34]:
import random
random.seed(123)   # 다른 시드로 새로운 5개 뽑기

sample_indices = random.sample(range(len(kor_corpus)), 5)

results = []
print("=" * 70)
print("무작위 샘플 5개")
print("=" * 70)

for idx in sample_indices:
    ko_sentence = kor_corpus[idx]
    en_reference = eng_corpus[idx]
    result_text, _, _, _ = evaluate(ko_sentence, model_bahdanau, encoder_tokenizer, decoder_tokenizer, max_len=45)

    results.append((idx, ko_sentence, en_reference, result_text))

    print(f"[{idx}]")
    print(f"  입력(한국어): {ko_sentence}")
    print(f"  정답(원문):   {en_reference}")
    print(f"  모델 예측:    {result_text}")
    print()

무작위 샘플 5개
[6863]
  입력(한국어): 또한 미국은 북한의 요청으로 , 경제 제재 완화 논의를 수 차례 준비했다 .
  정답(원문):   washington is also reportedly prepared to discuss easing financial sanctions , as requested by pyongyang .
  모델 예측:    also , top u . s . economic policies , too long and sanctions against north korea .

[35084]
  입력(한국어): 플로리다 주지사 , 상원의원 출마2009 . 06
  정답(원문):   florida s governor has decided not to run for re election .
  모델 예측:    florida , florida cnn florida s governor of florida , florida city has been getting back to rally last week .

[11427]
  입력(한국어): 그는 이제 더 이상 자연재해에 소홀히 대처해서는 안 된다 며 이 같은 불미스러운 사건은 더 이상 발생하지 않을 것이라고 약속할 수 없다 고 덧붙였다 .
  정답(원문):   never again will there be a mismanaged natural disaster , he said , later assuring the crowd that it will never happen again in this country you have my commitment and my promise .
  모델 예측:    i don t know that if you re not doing enough to do the same things that never again do worse , said yarrji , a further at the time of being a few or more aggressi

| 문장 정보 | 항목 | 평가 | 근거 |
|---|---|---|---|
| **[6863] 한국어**: 또한 미국은 북한의 요청으로, 경제 제재 완화 논의를 수 차례 준비했다. | | | |
| **정답**: washington is also reportedly prepared to discuss easing financial sanctions , as requested by pyongyang . | | | |
| **예측**: also , top u . s . economic policies , too long and sanctions against north korea . | | | |
| | 의미 전달 | ◐ 보통 | u.s., economic, sanctions, north korea 키워드는 등장하나 "논의를 준비했다"는 행위 자체는 빠짐 |
| | 문법 정확성 | ○ 나쁨 | "too long and sanctions against"에서 논리적 연결이 깨짐 |
| | 문장 구조 | ○ 나쁨 | 원문의 "주어+목적" 구조가 사라지고 명사 나열식으로 붕괴 |
| | 반복/붕괴 | ◐ 보통 | 단어 반복은 없으나 문장이 중간에 방향을 잃음 |
| | 숫자/고유명사 | ● 좋음 | u.s., north korea 정확히 대응 |
| | 문장 종결 | ● 좋음 | 짧게 마무리됨 |
| | 유창성 | ○ 나쁨 | "too long and sanctions"가 문법적으로 어색 |
| **[35084] 한국어**: 플로리다 주지사, 상원의원 출마 2009.06 | | | |
| **정답**: florida s governor has decided not to run for re election . | | | |
| **예측**: florida , florida cnn florida s governor of florida , florida city has been getting back to rally last week . | | | |
| | 의미 전달 | ○ 나쁨 | "재선 불출마"라는 핵심 사실이 전혀 없음, 오히려 반대 뉘앙스 |
| | 문법 정확성 | ○ 나쁨 | "florida cnn florida s governor of florida" 구간 명사 나열로 문법 파괴 |
| | 문장 구조 | ○ 나쁨 | 원문 구조 완전히 소실 |
| | 반복/붕괴 | ○ 나쁨 | "florida"가 4번 반복되는 전형적 붕괴 패턴 |
| | 숫자/고유명사 | ◐ 보통 | "florida"는 맞지만 과도하게 반복 |
| | 문장 종결 | ◐ 보통 | 끝나긴 하나 "last week"으로 뜬금없이 마무리 |
| | 유창성 | ○ 나쁨 | 전혀 읽히지 않는 수준 |
| **[11427] 한국어**: 그는 이제 더 이상 자연재해에 소홀히 대처해서는 안 된다며 이 같은 불미스러운 사건은 더 이상 발생하지 않을 것이라고 약속할 수 없다고 덧붙였다. | | | |
| **정답**: never again will there be a mismanaged natural disaster , he said , later assuring the crowd that it will never happen again in this country you have my commitment and my promise . | | | |
| **예측**: i don t know that if you re not doing enough to do the same things that never again do worse , said yarrji , a further at the time of being a few or more aggressive and should be allowed any | | | |
| | 의미 전달 | ○ 나쁨 | "다시는 없을 것을 약속한다"는 핵심 메시지가 전달 안 됨 |
| | 문법 정확성 | ○ 나쁨 | "never again do worse", "a further at the time of" 등 비문 다수 |
| | 문장 구조 | ○ 나쁨 | 마침표 없이 끝까지 이어지는 run-on 문장 |
| | 반복/붕괴 | ○ 나쁨 | 원문과 전혀 다른 화자 시점("i don't know that if...")으로 표류 |
| | 숫자/고유명사 | ○ 나쁨 | "yarrji"라는 존재하지 않는 이름을 생성(hallucination) |
| | 문장 종결 | ○ 나쁨 | eos 없이 max_len에서 강제 종료된 것으로 보임 |
| | 유창성 | ○ 나쁨 | 전혀 읽을 수 없는 수준 |
| **[53377] 한국어**: 케냐의 리프트밸리 중심도시 병원에 입원한 한 남성은 머리가 2차례 칼에 찔리고 손에 여러번 상처를 입었지만 살아났다. | | | |
| **정답**: lying in a hospital bed in this rural hub of kenya s rift valley , a man describes surviving two machete wounds to his head and multiple slashes to his hands . | | | |
| **예측**: at the hospital , a two men dressed in a kenya village hospital in rural kenya town of skia , kenya s rift valley , was briefly two machetes and two men and several people . | | | |
| | 의미 전달 | ◐ 보통 | hospital, kenya, rift valley, machete 등 핵심 소재는 살아있음 |
| | 문법 정확성 | ○ 나쁨 | "a two men", "was briefly two machetes" 등 단복수·구조 오류 다수 |
| | 문장 구조 | ◐ 보통 | 병원-케냐-리프트밸리 나열 구조는 원문과 유사하게 배치됨 |
| | 반복/붕괴 | ◐ 보통 | "kenya" 3회, "two" 3회 등장하며 다소 반복적 |
| | 숫자/고유명사 | ● 좋음 | kenya, rift valley 정확히 매칭(단, 존재하지 않는 지명 "skia" 생성) |
| | 문장 종결 | ● 좋음 | 자연스럽게 마침 |
| | 유창성 | ○ 나쁨 | 단복수 불일치로 어색함 |
| **[34937] 한국어**: 자살폭탄테러로 화상을 입은 아와미 국민당 소속 압둘 와히드(22)는 테러범이 이슬람경전인 코란의 한 구절을 인용하며 테러를 자행했다고 전했다. | | | |
| **정답**: abdul waheed , 22 , who suffered burns from the blast , said the bomber struck as a member of the party was reciting verses from islam s holy book , the quran . | | | |
| **예측**: abdul waheed , 22 , who was standing behind the plot , said the 22 year old woman who was standing behind the plot , and the man who left the dead and the attack , and the pope has been missing . | | | |
| | 의미 전달 | ● 좋음 | "abdul waheed , 22"가 정확히 일치, burns(화상) 정보는 반영 안 됐지만 인물 정보는 정확 |
| | 문법 정확성 | ◐ 보통 | 앞부분은 매끄러우나 뒷부분("the pope has been missing")이 뜬금없음 |
| | 문장 구조 | ◐ 보통 | "이름, 나이, who ... said" 구조까지는 원문과 거의 일치 |
| | 반복/붕괴 | ○ 나쁨 | "who was standing behind the plot"이 두 번 반복, "22"도 중복 등장 |
| | 숫자/고유명사 | ● 좋음 | 이름과 나이가 완벽히 정확 (5개 중 최고) |
| | 문장 종결 | ◐ 보통 | 끝나긴 하나 원문에 없는 내용("교황")으로 마무리 |
| | 유창성 | ◐ 보통 | 전반부는 자연스럽고 후반부만 붕괴 |

### 무작위 샘플 5개 정성적 평가 점수

| 문장 | 의미전달 | 문법 | 구조 | 반복/붕괴 | 숫자/고유명사 | 종결 | 유창성 | 합계 | 정성 점수(%) |
|---|---|---|---|---|---|---|---|---|---|
| [6863] 미국-북한 제재 | 1 | 0 | 0 | 1 | 2 | 2 | 0 | 6/14 | 42.9% |
| [35084] 플로리다 주지사 | 0 | 0 | 0 | 0 | 1 | 1 | 0 | 2/14 | 14.3% |
| [11427] 자연재해 약속 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0/14 | **0.0%** |
| [53377] 케냐 병원 | 1 | 0 | 1 | 1 | 2 | 2 | 0 | 7/14 | 50.0% |
| [34937] 압둘 와히드 | 2 | 1 | 1 | 0 | 2 | 1 | 1 | 8/14 | **57.1%** |
| **평균** | | | | | | | | | **32.9%** |

### 무작위 샘플 5개 정량적 평가(BLEU) 점수와 정성 점수 비교

| 문장 | 정성 점수 | BLEU | 순위 일치 여부 |
|---|---|---|---|
| [34937] 압둘 와히드 | 57.1% (1위) | 0.175 (1위) | ✅ 일치 |
| [53377] 케냐 병원 | 50.0% (2위) | 0.171 (2위) | ✅ 일치 |
| [6863] 미국-북한 | 42.9% (3위) | 0.086 (4위) | ❌ 불일치 |
| [35084] 플로리다 | 14.3% (4위) | 0.124 (3위) | ❌ 불일치 |
| [11427] 자연재해 | 0.0% (5위) | 0.053 (5위) | ✅ 일치 |

#### 두 가지 발견
###### 1)두 지표가 극단적으로 갈리는 지점 [11427] 문장은 정성 점수 0점(7개 항목 전부 최하)인데,
###### BLEU는 0.053으로 완전히 0은 문장이 워낙 길어서(42단어) 우연히 겹치는 1-gram 단어(관사, 전치사 등)가 있음.
###### 이 겹침이 "의미 있는 정보 전달"과는 무관함. (예: "the", "a", "that" 같은 기능어끼리의 우연한 일치)
###### 
###### 2)반대로 [35084]는 정성 점수가 14.3%로 거의 바닥인데, BLEU는 0.124로 중간 수준.
###### "florida" 반복이 재현율을 인위적으로 끌어올린 것이 원인임.
###### 
#### 결론
###### 자동화된 정량 지표만으로는 실제 번역 품질(특히 붕괴되거나 반복적인 문장)을 정확히 포착하기 어려우며,
###### 사람이 직접 판단하는 정성적 평가가 병행되어야 한다.

##### 

### 📔 문장의 언어학적 평가

##### 문장의 어떠한 요소가 문장의 의미(진리조건, truth condition)를 결정하는 데 핵심적인 역할을 하는가

| 순위 | 요소 | 왜 치명적인가 |
|---|---|---|
| 1 | 부정(not/안/못) | 진리값을 정반대로 뒤집음 |
| 2 | 서술어(동사) | "무슨 사건인지" 자체를 결정 |
| 3 | 주어-목적어 관계(조사/어순) | "누가 누구에게"가 뒤바뀜 |
| 4 | 수량사/숫자 | 정확한 사실관계가 어긋남 |
| 5 | 시제 | 사건은 같으나 "언제"가 달라짐 |
| 6 | 수식어(형용사/부사) | 부가 정보 손실, 핵심 골격은 유지됨 |

##### -> 현실적인 제약 조건으로 수량사/숫자가 seq2seq 번역기에서 잘 나오는지 확인해보기

#### 1) 수량사 하나만 나온 문장 

In [43]:
import re
import random

def count_numbers(sentence):
    """숫자 덩어리(연속된 숫자)의 개수를 센다."""
    return len(re.findall(r'\d+', sentence))

# 지금 시점의 kor_corpus 기준으로 필터링
one_number_indices = [i for i in range(len(kor_corpus)) if count_numbers(kor_corpus[i]) == 1]
print(f"숫자 1개 문장 개수: {len(one_number_indices)}개")

random.seed(7)
sample_one = random.sample(one_number_indices, 5)

print("=" * 70)
print("숫자 1개 포함 문장 5개")
print("=" * 70)

for idx in sample_one:
    ko_sentence = kor_corpus[idx]
    en_reference = eng_corpus[idx]
    result_text, _, _, _ = evaluate(ko_sentence, model_bahdanau, encoder_tokenizer, decoder_tokenizer, max_len=45)

    # 재확인: 이 문장에 정말 숫자가 1개인지 즉석에서 검증
    actual_count = count_numbers(ko_sentence)

    print(f"[{idx}] (숫자 개수 확인: {actual_count})")
    print(f"  입력(한국어): {ko_sentence}")
    print(f"  정답(원문):   {en_reference}")
    print(f"  모델 예측:    {result_text}")
    print()

숫자 1개 문장 개수: 15190개
숫자 1개 포함 문장 5개
[24997] (숫자 개수 확인: 1)
  입력(한국어): 코소보의 영토는 북대서양조약기구 가 서브족의 군력 추방 , 살해 중지 , 알바니아인의 추방을 수행했던 1999년부터 미국이 지배하고 있다 .
  정답(원문):   the territory has been run by the united nations since 1999 , when nato carried out a bombing campaign to drive out serb forces and halt the killing and expulsion of albanians during a two year separatist war .
  모델 예측:    the population has sparked the territory , which is backed by an islamist ethnic albanian army , which has targeted the civil war with russia to confront it falls , which killed the anniversary during the century war , which killed the last two years

[11597] (숫자 개수 확인: 1)
  입력(한국어): 반면 샤라포바는 비너스의 첫 서브 평균 속도는 185 였으며 그것은 오늘 경기에서 내가 가장 빠른 서비스를 구사한 것과 같은 속도 라고 지적했다 .
  정답(원문):   sharapova conceded that venus was just too good for her on the day . she was averaging her first serve 115 miles per hour , where my first serve , the fastest one was 115 , sharapova said .
  모델 예측:    it was just for just below the first

#### 결과 분석

| 문장 | 정답 숫자 | 예측 숫자 | 판정 | 세부 내용 |
|---|---|---|---|---|
| [24997] | 1999 | (없음) | ❌ 생략 | 연도 정보가 통째로 사라짐 |
| [11597] | 115 (두 번 언급) | (없음) | ❌ 생략 | 서브 속도(115마일) 완전히 누락 |
| [30610] | 2005 | 364, 36 | ❌ 왜곡 | 연도가 엉뚱한 숫자(364, 36)로 대체됨 |
| [50296] | 8 | 8 (두 번 등장) | ✅ 정확 | 유일하게 숫자가 정확히 살아남음 |
| [3717] | 200 | (없음) | ❌ 생략 | 인원수(200여명)가 완전히 사라짐 |

#### 종합적인 패턴 

| 숫자 유형 | 경향 |
|---|---|
| "N명 사망/체포" 같은 사상자 수 | 비교적 정확 (8명 사례처럼) 또는 완전 생략, 중간이 별로 없음 |
| 연도(YYYY) | 생략되거나 완전히 엉뚱한 숫자로 왜곡됨 (가장 취약) |
| 속도/수치(mph 등) | 완전히 생략되는 경향 |

#### 2) 수량사 두 개 이상 문장

In [53]:
import re
import random

# 영어 숫자 단어 목록
NUMBER_WORDS = {
    'zero','one','two','three','four','five','six','seven','eight','nine','ten',
    'eleven','twelve','thirteen','fourteen','fifteen','sixteen','seventeen',
    'eighteen','nineteen','twenty','thirty','forty','fifty','sixty','seventy',
    'eighty','ninety','hundred','thousand','million','billion','dozen'
}

def extract_numbers_extended(text):
    """아라비아 숫자 + 영어 숫자 단어를 모두 추출해서 리스트로 반환"""
    digit_matches = re.findall(r'\d+', text)
    word_matches = [w for w in re.findall(r'[a-zA-Z]+', text.lower()) if w in NUMBER_WORDS]
    return digit_matches + word_matches

def count_korean_numbers(text):
    return len(re.findall(r'\d+', text))

def count_numbers_extended(text):
    return len(extract_numbers_extended(text))

In [54]:
# 한국어와 영어 정답 둘 다 숫자가 2개 이상 있는 문장만 필터링
strict_multi_indices = [
    i for i in range(len(kor_corpus))
    if count_korean_numbers(kor_corpus[i]) >= 2 and count_numbers_extended(eng_corpus[i]) >= 2
]

print(f"한국어·영어 모두 숫자 2개 이상인 문장 개수: {len(strict_multi_indices)}개")

random.seed(21)
sample = random.sample(strict_multi_indices, 5)

print("=" * 70)
print("숫자 2개 이상 (한국어+영어 둘 다 확인된) 문장 5개")
print("=" * 70)

for idx in sample:
    ko_sentence = kor_corpus[idx]
    en_reference = eng_corpus[idx]
    result_text, _, _, _ = evaluate(ko_sentence, model_bahdanau, encoder_tokenizer, decoder_tokenizer, max_len=45)

    ko_count = count_korean_numbers(ko_sentence)
    en_count = count_numbers_extended(en_reference)

    ref_numbers = extract_numbers_extended(en_reference)
    pred_numbers = extract_numbers_extended(result_text)

    print(f"[{idx}] (한국어 숫자: {ko_count}개, 영어 정답 숫자: {en_count}개)")
    print(f"  입력(한국어): {ko_sentence}")
    print(f"  정답(원문):   {en_reference}")
    print(f"  모델 예측:    {result_text}")
    print(f"  정답 숫자들:  {ref_numbers}")
    print(f"  예측 숫자들:  {pred_numbers}")
    print()

한국어·영어 모두 숫자 2개 이상인 문장 개수: 8881개
숫자 2개 이상 (한국어+영어 둘 다 확인된) 문장 5개
[22138] (한국어 숫자: 2개, 영어 정답 숫자: 2개)
  입력(한국어): 선터카운티 보안관 피트 스미스는 최소 7명이 중경상을 입었으며 50명의 환자들이 병원에서 도시 근교의 다른 의료 시설로 이송됐다고 말했다 .
  정답(원문):   sumter county sheriff pete smith said at least seven people were critically injured . in all , 50 patients were evacuated from the hospital to other medical facilities in nearby towns , he said .
  모델 예측:    summary , california cnn police in order were killed in 184 , three medical doctors , several people were injured in the accident with other people in the accident , including several other people being treated , the statement said .
  정답 숫자들:  ['50', 'seven']
  예측 숫자들:  ['184', 'three']

[55320] (한국어 숫자: 8개, 영어 정답 숫자: 8개)
  입력(한국어): 메릴린치가 지난 1분기 서브프라임 손실로 60 80억 달러의 자산을 상각할 것이라고 소식에 엔 달러 환율은 1달러 당 102 . 04엔에서 100 . 84엔으로 떨어졌다 .
  정답(원문):   the dollar slipped to 100 . 84 japanese yen from 102 . 04 yen following reports in the media that merrill lynch co . will announce 6 billion to 8

#### 결과 분석

| 문장 | 실제 한국어 숫자 개수 | 정답 숫자 | 예측 숫자 | 일치 개수 | 판정 |
|---|---:|---|---|---|---|
| [22138] 선터카운티 보안관 | 2개 | 50, seven | 184, three | 0/2 | ❌ 완전 불일치 (숫자 두 개 모두 다른 값으로 대체) |
| [55320] 메릴린치 환율 | 8개 | 100, 84, 102, 04, 6, 8, billion, billion | 100, 000, 100, 100, 000, 100 | 1/8 | ❌ 대부분 누락, "100"만 우연히 반복 등장, 나머지(84, 102, 6, 8 등) 전부 소실 |
| [55322] URL/기사코드 | 2개 | 22, 3170519 | 22, 31519583195 | 1/2 | ⚠️ 부분 일치 ("22"는 정확, 큰 숫자는 자릿수까지 완전히 다른 값으로 왜곡) |
| [37291] 글렌 재규어 발견 | 2개 | 2006, 40, one | 2006, 0, 45 | 1/3 | ⚠️ 부분 일치 (연도 "2006"만 정확, 거리·개수 정보는 전부 틀림) |
| [63274] 초고속 열차 개통 | 2개 | 10, 40, four, two | 30, 000 | 0/4 | ❌ 완전 불일치 (정답 숫자 4개 중 하나도 못 맞힘) |

#### 숫자 1개, 2개 이상 문장 비교 분석

| 그룹 | 정확도 | 주요 실패 양상 |
|---|---|---|
| 숫자 1개 | 20% (1/5) | 생략(3건), 왜곡(1건) |
| 숫자 2개 이상 | 0% (0/4, 유효 샘플 기준) | 대부분 누락(1건), 완전 불일치(2건), 부분 일치이나 왜곡 동반(1건), 없는 숫자 창작(hallucination) |

### 🍦결론 

#### Q. 왜 숫자가 있는 문장에서 이러한 오류가 일어날까? 

##### 1.Hallucination(할루시네이션)

###### -숫자 토큰은 서로 다른 임베딩을 가지지만 문장 안에서는 동일한 문법적 역할을 한다.

###### 예: 17세, 21세, 29세 -> 각각 다른 의미이지만 문장 안에서는 위치, 문법적 역할이 동일함

###### 이 자리에 숫자가 와야 한다는 패턴은 확실히 배우지만, 정확히 몇 이라는 구체적인 값은 서로 헷갈리기 쉽게 학습됨

###### 

###### -Teacher forcing: 학습할 때와 번역을 만들 때 차이 

###### 학습 중에는 제대로 학습하지 못하는 자리에 강제로 정답을 넣어준다->teacher forcing 선생님의 강요(이게 정답이야!!)

###### 실제 추론에는 모델이 자기가 방금 만든 단어를 다음 입력으로 입력. 즉 오답이 더 오답이 된다.(Exposrue bias)

###### 

###### -SentencePiece: 긴 숫자를 여러 조각(서브워드)로 쪼갬

###### 17693은 하나의 숫자가 아니라, 여러 조각으로 나누어 있을 수 있음. 디코더가 이 조각들을 순차적으로 붙이다

###### 이쯤에서 멈춰야지를 놓치면 숫자가 계속 이어 붙는 결과가 된다. 

In [49]:
print(decoder_tokenizer.encode("17693", out_type=str))

['▁17', '6', '9', '3']


#### Q. 왜 숫자가 두 개 이상일 때 이러한 현상이 더 심해지는가?

###### -Attention이 어느 숫자를 봐야 하는지 헷갈림 
###### 디코더가 숫자를 만들 때 인코더 쪽의 특정 위치에 집중을 해야 정확한 숫자를 가져올 수 있음
###### 숫자가 하나뿐이면 attention이 어디를 봐야 할지 안 헷갈림. 근데 2개 이상이면, 인코더 시퀀스 안에서 가까이 붙어있을 수 있음
###### 예: [25643] 정답: 2003, 2006 → 예측엔 2006만 남음
###### 비슷한 종류의 정보(숫자, 특히 같은 4자리 연도)가 여러 개 나란히 있으면 서로 간섭을 일으킴
###### 
###### -Sequence 안에서 숫자가 여러 개 등장하면 숫자 자체를 찾는 것뿐 아니라 각 숫자와 해당 문맥의 관계를 유지하는 게 어려움
###### 예: [31479]에서 숫자가 8개나 쏟아진 게 현 상황임
###### 한 번 숫자 토큰을 생성하기 시작하면, 언제 빠져나와야 할지 판단을 못 하고 계속 숫자를 뱉어내는 것

##### 끝